# MScFE 690 Capstone — Task 2
## Data Pipeline, Validation, Initial Benchmark Portfolios and Backtesting

**Research universe:** SPY, EFA, EEM, IEF, TLT, LQD, GLD, DBC  
**Data frequency:** Daily  
**Sample:** 2010-01-01 to 2025-12-31  
**Prediction horizon:** Next 5 trading days  
**Portfolio rebalance frequency:** Weekly  

This notebook is designed to be run directly in **Google Colab**.


In [ ]:
# Install packages in Colab
!pip -q install yfinance cvxpy openpyxl


In [ ]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import cvxpy as cp

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## 1. Configuration

Yahoo Finance treats the `end` date as exclusive, so `2026-01-01` is used to include observations through `2025-12-31`.


In [ ]:
TICKERS = ["SPY", "EFA", "EEM", "IEF", "TLT", "LQD", "GLD", "DBC"]

START_DATE = "2010-01-01"
END_DATE = "2026-01-01"

TRAIN_END = "2016-12-31"
VALID_START = "2017-01-01"
VALID_END = "2018-12-31"
TEST_START = "2019-01-01"

MAX_WEIGHT = 0.30
LOOKBACK_DAYS = 252
WEEKLY_RULE = "W-FRI"

BASE = Path(".")
for p in [
    BASE / "data" / "raw",
    BASE / "data" / "processed",
    BASE / "data" / "validation",
    BASE / "outputs" / "benchmarks",
    BASE / "outputs" / "figures",
]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready.")


## 2. Download Daily Yahoo Finance Data


In [ ]:
raw = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    group_by="column",
    progress=False,
    threads=True,
)

print("Raw shape:", raw.shape)
display(raw.head())


In [ ]:
# Save raw download
raw.to_csv(BASE / "data" / "raw" / "yahoo_etf_raw.csv")
print("Saved raw Yahoo Finance data.")


## 3. Extract Adjusted Close Prices

Adjusted close prices are used because they account for distributions and corporate actions.


In [ ]:
if isinstance(raw.columns, pd.MultiIndex):
    if "Adj Close" in raw.columns.get_level_values(0):
        adjusted_close = raw["Adj Close"].copy()
    else:
        raise KeyError("Adj Close was not returned by Yahoo Finance.")
else:
    raise ValueError("Unexpected Yahoo Finance column structure.")

adjusted_close = adjusted_close.reindex(columns=TICKERS).sort_index()
adjusted_close.index = pd.to_datetime(adjusted_close.index)

display(adjusted_close.head())
print(adjusted_close.shape)


## 4. Data Validation

The checks below cover:
- first and last valid observation
- observation counts
- missing values
- duplicate dates
- non-positive adjusted prices
- suspiciously large daily returns


In [ ]:
validation_rows = []

for ticker in TICKERS:
    s = adjusted_close[ticker]

    validation_rows.append({
        "Ticker": ticker,
        "First_Valid_Date": s.first_valid_index(),
        "Last_Valid_Date": s.last_valid_index(),
        "Observations": int(s.notna().sum()),
        "Missing_Values": int(s.isna().sum()),
        "Non_Positive_Prices": int((s.dropna() <= 0).sum()),
    })

quality_report = pd.DataFrame(validation_rows)

duplicate_dates = int(adjusted_close.index.duplicated().sum())

print("Duplicate dates in index:", duplicate_dates)
display(quality_report)


In [ ]:
# Daily returns used only for diagnostic validation at this stage
diagnostic_returns = adjusted_close.pct_change(fill_method=None)

extreme_return_report = pd.DataFrame({
    "Min_Daily_Return": diagnostic_returns.min(),
    "Max_Daily_Return": diagnostic_returns.max(),
    "Abs_Return_Above_20pct": (diagnostic_returns.abs() > 0.20).sum()
}).reset_index().rename(columns={"index": "Ticker"})

quality_report = quality_report.merge(extreme_return_report, on="Ticker", how="left")
quality_report["Duplicate_Dates_In_Index"] = duplicate_dates

display(quality_report)
quality_report.to_csv(BASE / "data" / "validation" / "data_quality_report.csv", index=False)


## 5. Align the Investment Universe

For the core modelling dataset, use only dates where all eight ETFs have valid adjusted prices.

This avoids artificial zero-filling and ensures every portfolio decision is based on a common cross-section.


In [ ]:
prices_aligned = adjusted_close.dropna(how="any").copy()

print("Before alignment:", adjusted_close.shape)
print("After alignment :", prices_aligned.shape)
print("Start:", prices_aligned.index.min())
print("End  :", prices_aligned.index.max())

prices_aligned.to_csv(BASE / "data" / "processed" / "adjusted_close.csv")


## 6. Daily Simple Returns and Log Returns


In [ ]:
daily_returns = prices_aligned.pct_change(fill_method=None).dropna()
log_returns = np.log(prices_aligned / prices_aligned.shift(1)).dropna()

daily_returns.to_csv(BASE / "data" / "processed" / "daily_returns.csv")
log_returns.to_csv(BASE / "data" / "processed" / "log_returns.csv")

display(daily_returns.head())


## 7. Five-Trading-Day Forward Return Target

For asset `i` at time `t`:

\[
R_{i,t\rightarrow t+5} = \frac{P_{i,t+5}}{P_{i,t}} - 1
\]

This is a **forward-looking target**, not a backward five-day return.


In [ ]:
forward_5d_returns = prices_aligned.shift(-5) / prices_aligned - 1
forward_5d_returns.to_csv(BASE / "data" / "processed" / "forward_5d_returns.csv")

display(forward_5d_returns.head())


## 8. Research Period Labels


In [ ]:
calendar = pd.DataFrame(index=prices_aligned.index)

calendar["Period"] = np.select(
    [
        calendar.index <= pd.Timestamp(TRAIN_END),
        (calendar.index >= pd.Timestamp(VALID_START)) & (calendar.index <= pd.Timestamp(VALID_END)),
        calendar.index >= pd.Timestamp(TEST_START),
    ],
    ["Train", "Validation", "Test"],
    default="Outside"
)

display(calendar["Period"].value_counts())


## 9. Five-Trading-Day Purge at Sample Boundaries

Because the target uses the next five trading days, the last five trading observations immediately before the validation and test periods are flagged for exclusion from model fitting.


In [ ]:
calendar["Purged"] = False

valid_start_actual = calendar.index[calendar.index >= pd.Timestamp(VALID_START)][0]
test_start_actual = calendar.index[calendar.index >= pd.Timestamp(TEST_START)][0]

before_valid = calendar.index[calendar.index < valid_start_actual][-5:]
before_test = calendar.index[calendar.index < test_start_actual][-5:]

calendar.loc[before_valid, "Purged"] = True
calendar.loc[before_test, "Purged"] = True

calendar["Usable_For_Model"] = ~calendar["Purged"]

print("Validation starts:", valid_start_actual)
print("Test starts      :", test_start_actual)
print("\nPurged before validation:")
print(before_valid)
print("\nPurged before test:")
print(before_test)

calendar.to_csv(BASE / "data" / "processed" / "modelling_calendar.csv")


## 10. Weekly Portfolio Rebalance Dataset

The underlying information remains daily, but portfolio decisions are made weekly.


In [ ]:
weekly_prices = prices_aligned.resample(WEEKLY_RULE).last().dropna()
weekly_returns = weekly_prices.pct_change(fill_method=None).dropna()

weekly_prices.to_csv(BASE / "data" / "processed" / "weekly_prices.csv")
weekly_returns.to_csv(BASE / "data" / "processed" / "weekly_returns.csv")

print("Weekly prices:", weekly_prices.shape)
display(weekly_prices.head())


# 11. Initial Benchmark Portfolios

The capstone design specifies:
1. Equal Weight
2. Mean-Variance
3. Risk Parity

These benchmarks are intentionally independent of the later ML/CVaR framework.


In [ ]:
N_ASSETS = len(TICKERS)

def equal_weight():
    return pd.Series(
        np.repeat(1 / N_ASSETS, N_ASSETS),
        index=TICKERS,
        name="weight"
    )

equal_weight()


## 11.1 Mean-Variance Benchmark

A practical long-only minimum-variance benchmark is used here for the initial pipeline.

Constraints:
- fully invested
- long only
- maximum 30% per asset


In [ ]:
def minimum_variance_weights(cov_matrix, max_weight=MAX_WEIGHT):
    n = cov_matrix.shape[0]
    w = cp.Variable(n)

    objective = cp.Minimize(cp.quad_form(w, cov_matrix.values))
    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        w <= max_weight
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.CLARABEL, verbose=False)

    if w.value is None:
        return pd.Series(np.repeat(1/n, n), index=cov_matrix.columns)

    return pd.Series(np.asarray(w.value).flatten(), index=cov_matrix.columns)


## 11.2 Risk-Parity Benchmark

This implementation uses inverse volatility as a transparent initial risk-parity approximation.

A full equal-risk-contribution optimiser can be added later if required.


In [ ]:
def inverse_volatility_weights(cov_matrix, max_weight=MAX_WEIGHT):
    vol = np.sqrt(np.diag(cov_matrix.values))
    inv_vol = 1 / vol
    w = inv_vol / inv_vol.sum()

    # Iterative cap-and-redistribute for the 30% concentration limit
    w = pd.Series(w, index=cov_matrix.columns)

    for _ in range(20):
        over = w > max_weight
        if not over.any():
            break

        excess = (w[over] - max_weight).sum()
        w[over] = max_weight

        under = ~over
        if under.sum() == 0:
            break

        current_under = w[under].sum()
        if current_under > 0:
            w[under] += excess * (w[under] / current_under)

    return w / w.sum()


# 12. Walk-Forward Benchmark Backtest

To avoid look-ahead bias:
- weights at each weekly rebalance date are estimated using only historical daily returns
- a 252-trading-day lookback window is used
- the resulting weights are applied to the following weekly return


In [ ]:
rebalance_dates = weekly_prices.index
benchmark_returns = []
weight_records = []

for i in range(1, len(rebalance_dates)):
    decision_date = rebalance_dates[i-1]
    return_date = rebalance_dates[i]

    hist = daily_returns.loc[:decision_date].tail(LOOKBACK_DAYS)

    if len(hist) < 60:
        continue

    cov = hist.cov() * 252

    weights = {
        "Equal_Weight": equal_weight(),
        "Mean_Variance": minimum_variance_weights(cov),
        "Risk_Parity": inverse_volatility_weights(cov),
    }

    next_ret = weekly_returns.loc[return_date]

    row = {"Date": return_date}

    for name, w in weights.items():
        w = w.reindex(TICKERS).fillna(0)
        row[name] = float((w * next_ret).sum())

        for ticker, val in w.items():
            weight_records.append({
                "Decision_Date": decision_date,
                "Return_Date": return_date,
                "Portfolio": name,
                "Ticker": ticker,
                "Weight": float(val)
            })

    benchmark_returns.append(row)

benchmark_returns = pd.DataFrame(benchmark_returns).set_index("Date")
benchmark_weights = pd.DataFrame(weight_records)

display(benchmark_returns.head())


## 13. Benchmark Performance Metrics


In [ ]:
def max_drawdown(return_series):
    wealth = (1 + return_series).cumprod()
    peak = wealth.cummax()
    drawdown = wealth / peak - 1
    return drawdown.min()

def historical_cvar(return_series, alpha=0.95):
    losses = -return_series.dropna()
    var = losses.quantile(alpha)
    tail_losses = losses[losses >= var]
    return tail_losses.mean()

def performance_metrics(return_series, periods_per_year=52):
    r = return_series.dropna()

    ann_return = (1 + r).prod() ** (periods_per_year / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(periods_per_year)

    sharpe = np.nan
    if ann_vol > 0:
        sharpe = ann_return / ann_vol

    downside = r[r < 0].std(ddof=1) * np.sqrt(periods_per_year)
    sortino = np.nan if downside == 0 or np.isnan(downside) else ann_return / downside

    return pd.Series({
        "Annualized_Return": ann_return,
        "Annualized_Volatility": ann_vol,
        "Sharpe_Ratio": sharpe,
        "Sortino_Ratio": sortino,
        "Maximum_Drawdown": max_drawdown(r),
        "Historical_CVaR_95_Weekly": historical_cvar(r, 0.95)
    })

metrics = benchmark_returns.apply(performance_metrics).T
display(metrics)


In [ ]:
benchmark_returns.to_csv(
    BASE / "outputs" / "benchmarks" / "benchmark_weekly_returns.csv"
)

benchmark_weights.to_csv(
    BASE / "outputs" / "benchmarks" / "benchmark_weights.csv",
    index=False
)

metrics.to_csv(
    BASE / "outputs" / "benchmarks" / "benchmark_performance_metrics.csv"
)

print("Benchmark outputs saved.")


## 14. Cumulative Benchmark Performance Plot


In [ ]:
cumulative = (1 + benchmark_returns).cumprod()

ax = cumulative.plot(figsize=(12, 6), title="Initial Benchmark Portfolios")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("Date")
plt.tight_layout()
plt.savefig(BASE / "outputs" / "figures" / "benchmark_cumulative_performance.png", dpi=150)
plt.show()


## 15. Final Data Sanity Checks


In [ ]:
checks = pd.DataFrame({
    "Dataset": [
        "Adjusted Close",
        "Daily Returns",
        "Log Returns",
        "5-Day Forward Returns",
        "Weekly Prices",
        "Weekly Returns",
        "Benchmark Weekly Returns"
    ],
    "Rows": [
        len(prices_aligned),
        len(daily_returns),
        len(log_returns),
        len(forward_5d_returns),
        len(weekly_prices),
        len(weekly_returns),
        len(benchmark_returns)
    ],
    "Start": [
        prices_aligned.index.min(),
        daily_returns.index.min(),
        log_returns.index.min(),
        forward_5d_returns.index.min(),
        weekly_prices.index.min(),
        weekly_returns.index.min(),
        benchmark_returns.index.min()
    ],
    "End": [
        prices_aligned.index.max(),
        daily_returns.index.max(),
        log_returns.index.max(),
        forward_5d_returns.index.max(),
        weekly_prices.index.max(),
        weekly_returns.index.max(),
        benchmark_returns.index.max()
    ]
})

display(checks)


# 16. Files Produced

After a successful run, the important outputs are:

### Data
- `data/raw/yahoo_etf_raw.csv`
- `data/processed/adjusted_close.csv`
- `data/processed/daily_returns.csv`
- `data/processed/log_returns.csv`
- `data/processed/forward_5d_returns.csv`
- `data/processed/weekly_prices.csv`
- `data/processed/weekly_returns.csv`
- `data/processed/modelling_calendar.csv`

### Validation
- `data/validation/data_quality_report.csv`

### Benchmarks
- `outputs/benchmarks/benchmark_weekly_returns.csv`
- `outputs/benchmarks/benchmark_weights.csv`
- `outputs/benchmarks/benchmark_performance_metrics.csv`

### Figure
- `outputs/figures/benchmark_cumulative_performance.png`

The processed data can now be passed to the feature/model pipeline for the later ML and CVaR work.
